In [1]:
import tifffile as tiff
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

In [2]:
home_dir = os.path.expanduser("~")
data_dir = os.path.join(home_dir, "ext_hd_sammy", "data")
project_dir = os.path.join(home_dir, "ext_hd_sammy", "projects")

data_dir_comet = os.path.join(data_dir, "COMET")
data_dir_stomics = os.path.join(data_dir, "stomics")
data_dir_msi = os.path.join(data_dir, 'msi')

msi_glycans_path = os.path.join(data_dir_msi, "Glycan OMEtif files")
he_msi_path = os.path.join(data_dir_msi, "he")
if_comet_path = os.path.join(data_dir_comet, "original")
masks_comet_path = os.path.join(data_dir_comet, "visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples")
stomics_path = os.path.join(data_dir_stomics, "gene_exp")

out = os.path.join(project_dir, 'out')
out_matched_images_stom_comet = os.path.join(out, 'matched_images_stomics_COMET')
os.makedirs(out_matched_images_stom_comet, exist_ok=True)


sub_dir_path_to_stomics_dapi = "03.ssDNA_analysis"

meta_data_file_path = [os.path.join(data_dir,file) for file in os.listdir(data_dir) if file.endswith('.csv')][0]

In [3]:
stomics_dapi_paths = [os.path.join(stomics_path, d, f'{sub_dir_path_to_stomics_dapi}/ssDNA_{d}_regist.tif') for d in os.listdir(stomics_path) if os.path.isdir(os.path.join(stomics_path, d))]
he_msi_paths = [os.path.join(he_msi_path, d) for d in os.listdir(he_msi_path) if d.endswith('.ndpi')]
if_comet_paths = [os.path.join(if_comet_path, d) for d in os.listdir(if_comet_path) if d.endswith('.tiff')]
visio_comet_mask_paths = [os.path.join(masks_comet_path, d) for d in os.listdir(masks_comet_path) if d.endswith('.tif')]
glycan_paths = [os.path.join(msi_glycans_path, d) for d in os.listdir(msi_glycans_path) if d.endswith('.ome.tif')]

In [4]:
meta = pd.read_csv(meta_data_file_path)
meta.head(3)

,Sample ID,Chip ID,Aspera Folder,Stitching Errors,Passed ImageQC,Total Reads,Median MID/Bin200,Median Gene Type/ Bin200,Unique Reads,Sequencing saturation,Non-Relevant Short reads,cDNA Concentration,Library Concentration,Next Step Discussed During Meeting
0,SO1,D03453A6,Stomics_delivery,No,No,"1,031,513,686","7,410","4,224","67,555,776",48.2%,53.9%,23.00,25.8,"Manual registration was okay, newly prepared l..."
1,SO11,C03137E3,Stomics_delivery,No,Yes,"1,974,720,138","19,342","6,512","72,497,121",85.1%,32.3%,5.08,40.6,Morphology has been confirmed. Newly prepared ...
2,SO14,C03137D4,Stomics_delivery,No,Yes,"2,036,176,192","5,639","3,163","30,867,666",90.7%,47.1%,10.20,21.2,Gene capture observed likely due to significan...


In [5]:
len(stomics_dapi_paths), len(if_comet_paths), len(he_msi_paths), len(glycan_paths), len(visio_comet_mask_paths)

(40, 20, 32, 19, 11)

In [6]:
def normalize_id(name):
    return name.upper().replace('O', '0').strip()

In [7]:
### find matches for channel-msi, he-msi, if-comet, dapi-stomics
sample_ids = meta['Sample ID']
chip_ids = meta['Chip ID']

### clean up lists
valid_names = set(sample_ids).union(set(chip_ids))
stomics_dapi_paths = [file for file in stomics_dapi_paths if any(name in file for name in valid_names)]
if_comet_paths = [file for file in if_comet_paths if any(name in file for name in valid_names)]
he_msi_paths = [file for file in he_msi_paths if any(name in file for name in valid_names)]
glycan_paths = [file for file in glycan_paths if any(normalize_id(name) in file for name in valid_names)]
visio_comet_mask_paths = [file for file in visio_comet_mask_paths if any(name in file for name in valid_names)]

In [8]:
visio_comet_mask_paths

['/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO40_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO1_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO20(Aligned)_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO39_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO48_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO6_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO36_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_

In [9]:
len(stomics_dapi_paths), len(if_comet_paths), len(he_msi_paths), len(glycan_paths), len(visio_comet_mask_paths)

(35, 19, 26, 19, 10)

In [10]:
### only keep the elements that have a match

linked_samples = list(zip(sample_ids, chip_ids))
[print(sample_id, chip_id) for sample_id, chip_id in linked_samples]



SO1 D03453A6
SO11 C03137E3
SO14 C03137D4
SO15 C03137E6
SO4 C03027C4
SO5 C03027F5
SO6 C03036D6
SO8 C03030F4
SO16 C03137F1
SO17 D04159C4
SO18 D04165A2
SO19 D04165F6
SO20 D04164A1
SO21 D04161D1
SO22 A03979G5
SO23 D04159E4
SO24 D03451C5
SO25 C03450G3
SO26 C03449E3
SO27 C03450E6
SO28 D03452C6
SO29 D03452A6
SO30 D03453C2
SO31 C03449A4
SO32 D04160C4
SO33 D04165G2
SO34 A03979E2
SO36 C04139G6
SO37 C04138C1
SO38 C04138G4
SO39 C04140E2
SO40 C04143D3
SO44 B04101A3
SO45 A04100A6
SO46 C04139D3
SO47 C04138E4
SO48 B04106C3
SO50 C04138A5
SO51 C04140G4
SO52 C04142G4
SO53 C04140A3
SO54 C04143F3
SO55 C04141G2
SO56 C04143C6
SO57 C04138F3
SO58 C04139G2
SO59 C04143G2
SO61 B03715D3
SO63 B03710G3
SO64 B04101E6
SO65 B04106G5
SO66 B04101A5
SO67 B04106E6


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [11]:

glycan_paths_filtered = [file for file in glycan_paths if any(normalize_id(sample_id) in file or chip_id in file for sample_id, chip_id in linked_samples)]
if_comet_paths_filtered = [file for file in if_comet_paths if any(sample_id in file or chip_id in file for sample_id, chip_id in linked_samples)]
he_msi_paths_filtered = [file for file in he_msi_paths if any(sample_id in file or chip_id in file for sample_id, chip_id in linked_samples)]
stomics_dapi_paths_filtered = [file for file in stomics_dapi_paths if any(sample_id in file or chip_id in file for sample_id, chip_id in linked_samples)]
visio_comet_mask_paths_filtered = [file for file in visio_comet_mask_paths if any(sample_id in file or chip_id in file for sample_id, chip_id in linked_samples)]

In [12]:
visio_comet_mask_paths_filtered

['/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO40_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO1_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO20(Aligned)_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO39_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO48_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO6_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO36_BS.tif',
 '/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_

In [13]:
s0s_in_if = [name for name in sample_ids if any(name in file_path for file_path in if_comet_paths_filtered)]
s0s_in_if

['SO1',
 'SO14',
 'SO15',
 'SO4',
 'SO5',
 'SO6',
 'SO23',
 'SO24',
 'SO26',
 'SO32',
 'SO33',
 'SO34',
 'SO44',
 'SO45',
 'SO46',
 'SO50',
 'SO51',
 'SO58']

In [14]:
s0s_in_visio = [name for name in sample_ids if any(name in file_path for file_path in visio_comet_mask_paths_filtered)]
s0s_in_visio

['SO1',
 'SO4',
 'SO5',
 'SO6',
 'SO20',
 'SO36',
 'SO39',
 'SO40',
 'SO48',
 'SO53',
 'SO64',
 'SO67']

In [15]:
s0s_in_he = [name for name in sample_ids if any(name in file_path for file_path in he_msi_paths_filtered)]
s0s_in_he

['SO1',
 'SO14',
 'SO15',
 'SO4',
 'SO5',
 'SO6',
 'SO8',
 'SO19',
 'SO22',
 'SO23',
 'SO30',
 'SO36',
 'SO39',
 'SO40',
 'SO48',
 'SO50',
 'SO51',
 'SO52',
 'SO53',
 'SO54',
 'SO55',
 'SO56',
 'SO57',
 'SO58',
 'SO59',
 'SO64']

In [16]:
s0s_glycans = [name for name in sample_ids if any(normalize_id(name) in file_path for file_path in glycan_paths_filtered)]
s0s_glycans

['SO1',
 'SO4',
 'SO5',
 'SO6',
 'SO19',
 'SO20',
 'SO22',
 'SO23',
 'SO24',
 'SO25',
 'SO26',
 'SO30',
 'SO32',
 'SO33',
 'SO34',
 'SO44',
 'SO45',
 'SO46',
 'SO47']

In [17]:
s0_shared = set(s0s_in_he).intersection(set(s0s_in_if)).intersection(set(s0s_glycans))
s0_shared

{'SO1', 'SO23', 'SO4', 'SO5', 'SO6'}

In [18]:
meta_reduced = meta[meta['Sample ID'].isin(s0_shared)]
meta_reduced

,Sample ID,Chip ID,Aspera Folder,Stitching Errors,Passed ImageQC,Total Reads,Median MID/Bin200,Median Gene Type/ Bin200,Unique Reads,Sequencing saturation,Non-Relevant Short reads,cDNA Concentration,Library Concentration,Next Step Discussed During Meeting
0,SO1,D03453A6,Stomics_delivery,No,No,"1,031,513,686","7,410","4,224","67,555,776",48.2%,53.9%,23.00,25.8,"Manual registration was okay, newly prepared l..."
4,SO4,C03027C4,Stomics_delivery,Yes,Yes,"1,976,501,920","13,967","5,598","78,241,807",87.4%,23.3%,10.24,34.4,"Image has been restitched, newly prepared libr..."
5,SO5,C03027F5,Stomics_delivery,Yes,Yes,"2,624,619,504","17,954","6,292","122,313,254",85.3%,25.1%,16.70,28.4,"Image has been restitched, newly prepared libr..."
6,SO6,C03036D6,Stomics_delivery,No,Yes,"1,745,438,538","14,768","5,691","95,659,003",76.4%,41.0%,16.10,32.2,Newly prepared library to be sequenced for rem...
15,SO23,D04159E4,NaN,No,Yes,"1,596,704,180","2,260","1,509","18,967,208",93.30%,40.90%,14.60,16.9,Data from initial provided library and STOmics...


In [19]:
shared_chips = meta_reduced['Chip ID'].tolist()
chips_in_stomics = [name for name in shared_chips if any(name in file_path for file_path in stomics_dapi_paths_filtered)]
chips_in_stomics

['D03453A6', 'C03027C4', 'C03027F5', 'C03036D6', 'D04159E4']

In [21]:
import sys
import shutil

matched_ids = list(zip(meta_reduced['Sample ID'], meta_reduced['Chip ID']))

for sample_id, chip_id in matched_ids:
    print(sample_id, chip_id)
    
    
    he_matched = [fp for fp in he_msi_paths if os.path.basename(fp) == f'{sample_id}.ndpi']
    comet_matched = [fp for fp in if_comet_paths if os.path.basename(fp) == f'{sample_id}_BS.ome.tiff']
    stomics_matched = [fp for fp in stomics_dapi_paths if os.path.basename(fp) == f'ssDNA_{chip_id}_regist.tif']
    glycan_matched = [fp for fp in glycan_paths if os.path.basename(fp) == f'{normalize_id(sample_id)} ovary manual glycans.ome.tif']
    vis_matched = [fp for fp in visio_comet_mask_paths if os.path.basename(fp) == f'{sample_id}_BS.tif']
    
    ### load ans save only only channel for glycan
    g = tiff.imread(glycan_matched[0])
    g_single_channel = g[0,...]
    
    print(he_matched)
    print(comet_matched)
    print(stomics_matched)
    print(glycan_matched)
    print(vis_matched)
    
    if not os.path.exists(stomics_matched[0]):
        print(f"Stomics DAPI image not found for chip ID {chip_id}. Skipping this sample.")
        continue
    
    ### save each data in a new folder
    
    
    new_folder_path = os.path.join(out_matched_images_stom_comet, f'{sample_id}_{chip_id}')
    os.makedirs(new_folder_path, exist_ok=True)
    
    #tiff.imwrite(os.path.join(new_folder_path, f'{normalize_id(sample_id)}_glycans.ome.tif'), g_single_channel)
    #shutil.copy(he_matched[0], new_folder_path)
    shutil.copy(comet_matched[0], new_folder_path) 
    shutil.copy(stomics_matched[0], new_folder_path)
    shutil.copy(vis_matched[0], new_folder_path)
  
    

SO1 D03453A6
['/home/shamini_pathomics_io/ext_hd_sammy/data/msi/he/SO1.ndpi']
['/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/original/SO1_BS.ome.tiff']
['/home/shamini_pathomics_io/ext_hd_sammy/data/stomics/gene_exp/D03453A6/03.ssDNA_analysis/ssDNA_D03453A6_regist.tif']
['/home/shamini_pathomics_io/ext_hd_sammy/data/msi/Glycan OMEtif files/S01 ovary manual glycans.ome.tif']
['/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/visio_output/Vis_Analysis_All_Samples/Vis_Analysis_All_Samples/SO1_BS.tif']
SO4 C03027C4
['/home/shamini_pathomics_io/ext_hd_sammy/data/msi/he/SO4.ndpi']
['/home/shamini_pathomics_io/ext_hd_sammy/data/COMET/original/SO4_BS.ome.tiff']
['/home/shamini_pathomics_io/ext_hd_sammy/data/stomics/gene_exp/C03027C4/03.ssDNA_analysis/ssDNA_C03027C4_regist.tif']
['/home/shamini_pathomics_io/ext_hd_sammy/data/msi/Glycan OMEtif files/S04 ovary manual glycans.ome.tif']
[]
Stomics DAPI image not found for chip ID C03027C4. Skipping this sample.
SO5 C03027F5
['/home/shamini_